In [2]:
pip install langchain-chroma


  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
Using cached langchain_chroma-1.1.0-py3-none-any.whl (12 kB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
loader = TextLoader("speech.txt")
data = loader.load()
data

[Document(metadata={'source': 'speech.txt'}, page_content="Speech is the use of the human voice as a medium for language. Spoken language combines vowel and consonant sounds to form units of meaning like words, which belong to a language's lexicon. There are many different intentional speech acts, such as informing, declaring, asking, persuading, directing; acts may vary in various aspects like enunciation, intonation, loudness, and tempo to convey meaning. Individuals may also unintentionally communicate aspects of their social position through speech, such as sex, age, place of origin, physiological and mental condition, education, and experiences.\n\nWhile normally used to facilitate communication with others, people may also use speech without the intent to communicate. Speech may nevertheless express emotions or desires; people talk to themselves sometimes in acts that are a development of what some psychologists (e.g., Lev Vygotsky) have maintained is the use of silent speech in 

In [28]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
splits=text_splitter.split_documents(data)
splits

[Document(metadata={'source': 'speech.txt'}, page_content='Speech is the use of the human voice as a medium for language. Spoken language combines vowel and'),
 Document(metadata={'source': 'speech.txt'}, page_content='combines vowel and consonant sounds to form units of meaning like words, which belong to a'),
 Document(metadata={'source': 'speech.txt'}, page_content="which belong to a language's lexicon. There are many different intentional speech acts, such as"),
 Document(metadata={'source': 'speech.txt'}, page_content='acts, such as informing, declaring, asking, persuading, directing; acts may vary in various aspects'),
 Document(metadata={'source': 'speech.txt'}, page_content='in various aspects like enunciation, intonation, loudness, and tempo to convey meaning. Individuals'),
 Document(metadata={'source': 'speech.txt'}, page_content='Individuals may also unintentionally communicate aspects of their social position through speech,'),
 Document(metadata={'source': 'speech.txt'}, 

In [40]:
embeddings = OllamaEmbeddings(model="gemma:2b")


In [41]:
vectordb= Chroma.from_documents(documents=splits, embedding=embeddings)
vectordb

In [12]:
query="what does the traetment for brain problem?"
docs=vectordb.similarity_search(query)
docs[0].page_content

"to Broca's area, where morphology, syntax, and instructions for articulation are generated. This is then sent from Broca's area to the motor cortex for articulation.[30]"

In [13]:
vectordb = Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory="./chroma_db")


In [14]:
db2= Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
docs=db2.similarity_search(query)
docs[0].page_content

"to Broca's area, where morphology, syntax, and instructions for articulation are generated. This is then sent from Broca's area to the motor cortex for articulation.[30]"

In [15]:
docs = vectordb.similarity_search_with_score(query)
docs

[(Document(id='87b3cdcb-f385-4140-8905-abbd24888954', metadata={'source': 'speech.txt'}, page_content="to Broca's area, where morphology, syntax, and instructions for articulation are generated. This is then sent from Broca's area to the motor cortex for articulation.[30]"),
  2722.12646484375),
 (Document(id='78ff3913-268a-4b7c-a70a-56bc3ecad2dd', metadata={'source': 'speech.txt'}, page_content='eat-ate) remain unaffected.[34] Moreover, the circuits involved in human speech comprehension dynamically adapt with learning, for example, by becoming more efficient in terms of processing time when listening to familiar messages such as learned verses.[35]'),
  2748.95703125),
 (Document(id='12b4cdaa-160b-4967-8a53-6d7a444391b0', metadata={'source': 'speech.txt'}, page_content='and prosody but severe impairment in lexical access, resulting in poor comprehension and nonsensical or jargon speech.[31]'),
  2792.86279296875),
 (Document(id='c7e4eba1-f239-4a5f-b5e3-1027a47fda54', metadata={'sourc

In [16]:
retriever =vectordb.as_retriever()
retriever.invoke(query)[0].page_content

"to Broca's area, where morphology, syntax, and instructions for articulation are generated. This is then sent from Broca's area to the motor cortex for articulation.[30]"